# 1.DataSet Processing

In [1]:
import json
import os
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def load_from_json(filepath):
    """ Load a dictionary from a JSON file. """
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def save_to_json(filepath, data):
    """ Save a dictionary to a JSON file. """
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def preprocess_text(text):
    """Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
    tokens = word_tokenize(text.lower())

    #TODO: numbers in claims and evidence may be useful, use isanum() instead of isalpha()?
    # filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalpha() and word not in stop_words]
    filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
    return " ".join(filtered_tokens)

In [3]:
EVIDENCE_FILE = 'data/evidence.json'
TRAIN_FILE = 'data/train-claims.json'

evidence_data = load_from_json(EVIDENCE_FILE)
claims_data = load_from_json(TRAIN_FILE)

In [4]:
# # Map evidence IDs to their texts
# evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}

# # Check whether directory already exists
# path = 'data/curated'
# if not os.path.exists(path):
#   os.mkdir(path)

# # Save preprocessed evidence file
# save_to_json(path + '/processed_evidence_map.json', evidence_map)


In [5]:
evidence_map = load_from_json('data/curated/processed_evidence_map.json')
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur agricultu...
1,evidence-1,lindberg began profession career age 16 eventu...
2,evidence-2,boston ladi cambridg vampir weekend
3,evidence-3,gerald franci goyer born octob 20 1936 profess...
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...
...,...,...
1208822,evidence-1208822,also properti contribut garag apart
1208823,evidence-1208823,class fn org fyrd 6110 volda
1208824,evidence-1208824,dragon storm game game collect card game
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...


In [6]:
data_for_dataframe = []
for claim_id, claim_details in claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
claims_df = pd.DataFrame(data_for_dataframe)
claims_df 

,claim,evidence
0,scientif evid co2 pollut higher co2 concentr a...,"[evidence-442946, evidence-1194317, evidence-1..."
1,el niño drove record high global temperatur su...,"[evidence-338219, evidence-1127398]"
2,1946 pdo switch cool phase,"[evidence-530063, evidence-984887]"
3,weather channel john coleman provid evid convi...,"[evidence-1177431, evidence-782448, evidence-5..."
4,januari 2008 cap 12 month period global temper...,"[evidence-1010750, evidence-91661, evidence-72..."
...,...,...
1223,climat scientist say aspect case hurrican harv...,"[evidence-1055682, evidence-1047356, evidence-..."
1224,5th assess report 2013 ipcc estim human emiss ...,[evidence-916755]
1225,sinc mid 1970 global temperatur warm around de...,"[evidence-403673, evidence-889933, evidence-11..."
1226,abnorm temperatur spike februari earlier month...,"[evidence-97375, evidence-562427, evidence-521..."


In [7]:
import random

# Vectorization
vectorizer = TfidfVectorizer()
all_texts = claims_df['claim'].tolist() + evidence_df['evidence'].tolist()

# sample_texts = random.sample(all_texts, 500)

# # Fit the vectorizer on both claims and evidences
# vectorizer.fit(sample_texts) 
# claim_vec = vectorizer.transform(claims_df['claim']).toarray() 
# claims_df['claim_tfidf'] = list(claim_vec) 
# claim_vec.shape

In [8]:

# evidence_vec = vectorizer.transform(evidence_df['evidence']).toarray()
# evidence_vec.shape

In [10]:
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# # We need data for training the model
# processed_sentences = [sent.split() for sent in all_texts]

# model = Word2Vec(
#     sentences=processed_sentences,
# )


tagged_data = [TaggedDocument(words=_d.split(), tags=[str(i)]) for i, _d in enumerate(all_texts)]
tagged_data[:1]

[TaggedDocument(words=['scientif', 'evid', 'co2', 'pollut', 'higher', 'co2', 'concentr', 'actual', 'help', 'ecosystem', 'support', 'plant', 'anim', 'life'], tags=['0'])]

In [13]:
model = Doc2Vec(vector_size=50, min_count=1, epochs=20)
  
model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)
model.save("d2v.model")

In [15]:
model= Doc2Vec.load("d2v.model")

In [44]:
claims_df["vector"] = ""
for i in range(claims_df.shape[0]):
    inferred_vector = model.infer_vector(claims_df["claim"][i].split())
    claims_df["vector"][i] = inferred_vector
claims_df.to_csv('train_claim_vector.csv', index=False)
claims_df 

,claim,evidence,vector
0,scientif evid co2 pollut higher co2 concentr a...,"[evidence-442946, evidence-1194317, evidence-1...","[0.058819503, 0.20330563, -0.08398223, -0.1434..."
1,el niño drove record high global temperatur su...,"[evidence-338219, evidence-1127398]","[0.22781277, 0.01712014, -0.13577367, -0.15409..."
2,1946 pdo switch cool phase,"[evidence-530063, evidence-984887]","[0.022551605, 0.0755509, -0.11499794, 0.042379..."
3,weather channel john coleman provid evid convi...,"[evidence-1177431, evidence-782448, evidence-5...","[0.068567894, -0.111044094, -0.1413142, -0.010..."
4,januari 2008 cap 12 month period global temper...,"[evidence-1010750, evidence-91661, evidence-72...","[0.28036368, 0.23591448, -0.015527127, -0.1301..."
...,...,...,...
1223,climat scientist say aspect case hurrican harv...,"[evidence-1055682, evidence-1047356, evidence-...","[0.08556589, -0.024607938, -0.3162879, 0.03208..."
1224,5th assess report 2013 ipcc estim human emiss ...,[evidence-916755],"[0.022310777, 0.16586533, 0.028082661, -0.0328..."
1225,sinc mid 1970 global temperatur warm around de...,"[evidence-403673, evidence-889933, evidence-11...","[0.21405758, 0.14016956, -0.30209225, -0.05863..."
1226,abnorm temperatur spike februari earlier month...,"[evidence-97375, evidence-562427, evidence-521...","[0.009283262, -0.061513066, -0.18328325, -0.19..."


In [45]:
evidence_df["vector"] = ""
for i in range(evidence_df.shape[0]):
    inferred_vector = model.infer_vector(evidence_df["evidence"][i].split())
    evidence_df["vector"][i] = inferred_vector
evidence_df

,id,evidence,vector
0,evidence-0,john bennet law english entrepreneur agricultu...,"[-0.08329561, -0.15579118, -0.17553866, -0.129..."
1,evidence-1,lindberg began profession career age 16 eventu...,"[0.19000088, 0.15242222, -0.45844978, -0.01609..."
2,evidence-2,boston ladi cambridg vampir weekend,"[0.056133527, -0.1325368, 0.15981574, 0.088736..."
3,evidence-3,gerald franci goyer born octob 20 1936 profess...,"[0.12277374, -0.090537734, -0.3347684, -0.2872..."
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...,"[0.21676677, -0.151133, -0.48791853, 0.1081487..."
...,...,...,...
1208822,evidence-1208822,also properti contribut garag apart,"[0.08558074, -0.045330446, -0.14334618, -0.049..."
1208823,evidence-1208823,class fn org fyrd 6110 volda,"[0.07639017, -0.07667744, -0.3351644, -0.06447..."
1208824,evidence-1208824,dragon storm game game collect card game,"[0.14750834, -0.05903809, -0.40002766, -0.1349..."
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...,"[0.4817398, -0.25093532, -0.49334916, -0.16006..."


In [46]:
evidence_df.to_csv('evidence_vector.csv', index=False) 

In [43]:
inferred_vector = model.infer_vector(tagged_data[0].words)
model.dv.most_similar([inferred_vector], topn=len(model.dv))

[('0', 0.7956870794296265),
 ('125856', 0.7135351896286011),
 ('708463', 0.7124152779579163),
 ('1052984', 0.700192391872406),
 ('849012', 0.6973483562469482),
 ('1148229', 0.6943253874778748),
 ('1087808', 0.6826742887496948),
 ('121508', 0.6784234046936035),
 ('1030665', 0.6768766641616821),
 ('969570', 0.6707566380500793),
 ('1184353', 0.6688775420188904),
 ('109211', 0.665289044380188),
 ('988293', 0.6648860573768616),
 ('1198062', 0.6643795967102051),
 ('282080', 0.663949191570282),
 ('767015', 0.6629977226257324),
 ('36702', 0.6597950458526611),
 ('1061207', 0.6597950458526611),
 ('101005', 0.6594719886779785),
 ('1022742', 0.6588820219039917),
 ('211857', 0.6588616967201233),
 ('722709', 0.656978189945221),
 ('36181', 0.6567956805229187),
 ('747318', 0.6565003991127014),
 ('492540', 0.6552273631095886),
 ('411', 0.6549290418624878),
 ('753156', 0.6540385484695435),
 ('328872', 0.6530575752258301),
 ('1043592', 0.6523662805557251),
 ('787719', 0.6520460844039917),
 ('1105906', 0.

In [ ]:
# from gensim.models import KeyedVectors
# model.save("word2vec.model")
# word_vectors = model.wv
# word_vectors.save("word2vec.wordvectors")

# # Load back with memory-mapping = read-only, shared across processes.
# wv = KeyedVectors.load("word2vec.wordvectors", mmap='r')

In [47]:
DEV_FILE = 'data/dev-claims.json'
dev_claims_data = load_from_json(DEV_FILE)

data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

,claim,evidence
0,south australia expens electr world,"[evidence-67732, evidence-572512]"
1,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2..."
2,mean world 1c warmer time,"[evidence-889933, evidence-694262]"
3,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28..."
4,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947..."
...,...,...
149,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85..."
150,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20..."
151,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1..."
152,recent studi led lawrenc livermor nation labor...,[evidence-660755]


In [49]:
dev_claims_df["vector"] = ""
for i in range(dev_claims_df.shape[0]):
    inferred_vector = model.infer_vector(dev_claims_df["claim"][i].split())
    dev_claims_df["vector"][i] = inferred_vector

dev_claims_df 

,claim,evidence,vector
0,south australia expens electr world,"[evidence-67732, evidence-572512]","[-0.10937913, -0.18424094, 0.050457112, -0.180..."
1,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2...","[0.12197132, -0.19117312, -0.21191384, -0.1446..."
2,mean world 1c warmer time,"[evidence-889933, evidence-694262]","[-0.004584998, -0.24981353, -0.16938846, -0.08..."
3,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28...","[0.0804013, 0.361006, -0.12684847, 0.22560203,..."
4,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947...","[0.036465187, -0.04004745, -0.1368591, -0.0866..."
...,...,...,...
149,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85...","[0.23071034, -0.022975836, -0.004005174, -0.28..."
150,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20...","[0.10125896, -0.100763224, -0.04532265, -0.069..."
151,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1...","[-0.13560514, 0.1361969, -0.08570415, -0.05826..."
152,recent studi led lawrenc livermor nation labor...,[evidence-660755],"[0.36829528, 0.48349544, -0.24881983, -0.06897..."


In [59]:
dev_claims_df['vector'].to_numpy()[0]

array([-0.10937913, -0.18424094,  0.05045711, -0.18000066,  0.16202351,
        0.06514137, -0.10126383, -0.07224874,  0.01745951, -0.02079344,
        0.01686279, -0.0440724 ,  0.13147224,  0.11724794,  0.03221365,
       -0.0553623 ,  0.10530221,  0.08807132, -0.00030485, -0.04308194,
        0.08964672,  0.06640716,  0.1122754 , -0.00180796,  0.08492808,
       -0.02475946, -0.1432306 ,  0.03742294,  0.16611473, -0.16015913,
       -0.10248064,  0.2092451 , -0.12208004, -0.11941318,  0.14382598,
       -0.02406118, -0.06604342, -0.03854136, -0.05064041,  0.04722939,
        0.0794687 ,  0.03480593,  0.05887981, -0.00593496, -0.17539099,
       -0.06502956,  0.05654511, -0.0318765 , -0.02244504,  0.0960137 ],
      dtype=float32)

In [69]:
from sklearn.metrics.pairwise import cosine_similarity
X = np.array(dev_claims_df['vector'].values.tolist())
y = np.array(evidence_df['vector'].values.tolist())
sim = cosine_similarity(X, y)
sim
    

array([[ 0.13086517, -0.13080926,  0.03932882, ..., -0.04150811,
         0.00487678,  0.04380365],
       [-0.08061608,  0.81294066, -0.4080846 , ...,  0.83453006,
         0.67265135,  0.00724735],
       [ 0.05033674,  0.03039057,  0.00777365, ...,  0.10889159,
         0.19875602,  0.0981813 ],
       ...,
       [-0.0651614 , -0.04991781,  0.19374472, ..., -0.20686209,
        -0.03534394,  0.15142591],
       [-0.0534943 ,  0.1754826 , -0.19609277, ...,  0.1530917 ,
         0.02120426, -0.00655454],
       [-0.04026766,  0.12063422, -0.06529889, ...,  0.26122072,
         0.14257999,  0.18175806]], dtype=float32)

In [75]:
for i in range(sim.shape[0]):
    print(np.where(sim[i]>0.7))

(array([ 380025,  470376,  485602,  508446,  572512,  641060,  746852,
        827176,  882428,  939987, 1182886, 1198471, 1198787]),)
(array([      1,      11,      17, ..., 1208818, 1208822, 1208824]),)
(array([], dtype=int64),)
(array([ 130492,  200949,  260017,  274359,  455082,  576377,  715341,
        798037,  908707,  923818,  933654, 1055686]),)
(array([    295,     500,     853, ..., 1208055, 1208145, 1208147]),)
(array([   2614,    3113,    3506,    3833,    4198,    5868,    6760,
          7013,    7782,    9028,   10666,   10858,   11171,   11965,
         12161,   15189,   15382,   16043,   17566,   17583,   18061,
         18431,   18455,   18488,   19255,   19852,   21169,   21919,
         22038,   22090,   24434,   26652,   27445,   27788,   30172,
         30441,   34113,   34308,   34504,   36101,   36761,   38459,
         39662,   40712,   41473,   42412,   51458,   55604,   57537,
         57831,   59006,   59175,   62461,   62904,   64059,   66854,
         671